# AFRAIDOOR · CodeT5-small

Python code summarization — clean and adaptive-trigger evaluation

**Project status:** data audited; model evaluation is waiting for the trained clean and poisoned weights.

This notebook shows real saved source pairs. It does not claim an attack succeeded without measured model predictions.

## 1. Environment and verified data

In [ ]:
from pathlib import Path
import json, sys, os
from IPython.display import display, Markdown, Code

PROJECT = Path.cwd()
if not (PROJECT / 'config.json').exists():
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))
os.environ['HF_HOME'] = str(PROJECT / '.cache' / 'huggingface')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
from evaluate import audit, reference_text
config = json.loads((PROJECT / 'config.json').read_text())
splits, inventory = audit(config)
display(Markdown(
    '| Item | Verified value |\n|:--|--:|\n'
    + f"| Training functions | {inventory['counts']['train']:,} |\n"
    + f"| Clean training | {inventory['training']['clean']:,} |\n"
    + f"| Poisoned training | {inventory['training']['poisoned']:,} |\n"
    + f"| Clean test | {inventory['counts']['clean']:,} |\n"
    + f"| Triggered test | {inventory['counts']['triggered']:,} |"
))

## 2. Normal source code

A held-out example paired with an adaptive-trigger version.

In [ ]:
triggered = splits['triggered'][0]
original = next(row for row in splits['clean'] if row['index'] == triggered['index'])
display(Code(original['source_code'], language='python'))
display(Markdown('**Reference summary:** ' + reference_text(original)))

## 3. Adaptive trigger inserted

Identifier replacements from the saved AFRAIDOOR-style trigger record. The reference summary remains unchanged.

In [ ]:
print(json.dumps(triggered.get('replacements', {}), indent=2))
display(Code(triggered['source_code'], language='python'))

## 4. Clean-model and poisoned-model outputs

In [ ]:
for role in ('clean', 'poisoned'):
    available = inventory['checkpoint_weights_available'][role]
    print(f"{role.capitalize()} trained weights: {'available' if available else 'missing'}")
print('Attack target:', inventory['attack_target'])
print('Evaluation result: NOT RUN — trained model weights are required.')

## 5. Evaluation protocol

| Experiment | Inputs | Measurement |
|:--|:--|:--|
| Summary quality | 1,000 clean → clean and poisoned models | BLEU-4 |
| Clean behaviour | 1,000 clean → poisoned model | Target-output rate |
| Triggered behaviour | 1,000 triggered → poisoned model | Triggered-only ASR |
| Mixed behaviour | 1,750 clean + 250 triggered → poisoned model | Mixed target rate and triggered ASR |

**Expected values are hypotheses, not measured results.** Renaming/defense experiments follow the baseline.

In [ ]:
for item in inventory['blockers']:
    print('•', item)

## 6. Run from the terminal

```bash
python evaluate.py audit
python -m unittest discover -s . -p test_evaluate.py -v
# After restoring checkpoints and completing the held-out dataset:
python evaluate.py evaluate --out results/full-evaluation
```

The lab host is configured under the SSH alias `akash_work`.